# Module 5 · Solutions
Every answer ends with the honest sentence — what the numbers do and do not license you to say.

In [ ]:
import pandas as pd, numpy as np
import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
fin = pd.read_csv(BASE + "company_financials.csv")
cli = pd.read_csv(BASE + "client_book.csv", parse_dates=["onboard_date"])
uni = pd.read_csv(BASE + "nse_stock_universe.csv", parse_dates=["date"])

## 5A

In [ ]:
# Ex1 - volume split: stores vs same-store
fin["txns_cr"] = fin["revenue_cr"] / fin["avg_ticket_size_inr"]
y0 = fin[fin.fiscal_year=="FY24-25"].iloc[0]; y1 = fin[fin.fiscal_year=="FY25-26"].iloc[0]
ps0, ps1 = y0["txns_cr"]/y0["stores_count"], y1["txns_cr"]/y1["stores_count"]   # per-store volume
s0, s1 = y0["stores_count"], y1["stores_count"]
stores_effect    = (s1 - s0) * ps0        # new stores at old productivity
samestore_effect = (ps1 - ps0) * s0       # old stores doing more
interaction      = (s1 - s0) * (ps1 - ps0)
total = y1["txns_cr"] - y0["txns_cr"]
print(f"stores {stores_effect:+.3f} | same-store {samestore_effect:+.3f} | interaction {interaction:+.3f}")
print(f"sum {stores_effect+samestore_effect+interaction:+.3f} vs actual {total:+.3f} -> reconciles")
print("Store expansion is the engine; same-store growth is modest. Growth = footprint, not productivity.")

In [ ]:
# Ex2 - operating leverage in the COVID year
p, c_ = fin[fin.fiscal_year=="FY19-20"].iloc[0], fin[fin.fiscal_year=="FY20-21"].iloc[0]
rev_chg = c_["revenue_cr"]/p["revenue_cr"] - 1
print(f"Revenue: {rev_chg:+.1%}")
for col in ["cogs_cr","employee_cost_cr","marketing_cr","other_opex_cr"]:
    chg = c_[col]/p[col] - 1
    print(f"{col:<20} {chg:+7.1%}   flexibility = {chg/rev_chg:5.2f}")
print("COGS is nearly fully variable (flex ~1); employee cost is stickiest (flex near 0).")
print("Sticky costs on falling revenue = profit falls harder than sales: operating leverage, quantified.")

In [ ]:
# Ex3 - flipped Brinson weights
u25 = uni[uni["date"].dt.year==2025]
first = u25.sort_values("date").groupby("ticker")["close"].first()
last  = u25.sort_values("date").groupby("ticker")["close"].last()
ret = (last/first - 1).rename("ret").reset_index().merge(uni[["ticker","sector"]].drop_duplicates())
ret = ret[ret.sector.isin(["Financials","IT"])]
bench = ret.copy(); bench["w"] = 1/len(bench)
port = ret[ret.ticker.isin(["TCS.NS","INFY.NS","HDFCBANK.NS","ICICIBANK.NS"])].copy()
port["w"] = np.where(port.sector=="IT", 0.15, 0.35)     # FLIPPED: 70% Financials
def summ(df):
    return df.groupby("sector").apply(lambda x: pd.Series(
        {"w": x.w.sum(), "r": np.average(x.ret, weights=x.w)}), include_groups=False)
B, P = summ(bench), summ(port)
tb = (B.w*B.r).sum()
alloc = sum((P.loc[s,"w"]-B.loc[s,"w"])*(B.loc[s,"r"]-tb) for s in B.index)
sel   = sum(P.loc[s,"w"]*(P.loc[s,"r"]-B.loc[s,"r"]) for s in B.index)
print(f"Allocation {alloc:+.2%} (sign flips with the sector bet)")
print(f"Selection  {sel:+.2%} (moves little - picks are judged WITHIN sectors, insulated from the bet)")

## 5B

In [ ]:
# Ex1 - churn by cohort WITHIN tenure bands
cli["cohort"] = cli["onboard_date"].dt.year
cli["band"] = pd.cut(cli["tenure_months"], [0,24,48,84,144])
r = cli.pivot_table(index="cohort", columns="band", values="churned", aggfunc="mean", observed=True)
n = cli.pivot_table(index="cohort", columns="band", values="churned", aggfunc="count", observed=True)
print((r*100).round(0)); print("\ncounts:\n", n)
print("\nCohort and tenure are nearly aliases in a snapshot (2025 joiners CAN'T have 100-month tenure),")
print("so most cells are empty and the 'vintage story' largely collapses into a tenure story.")
print("Honest conclusion: this snapshot cannot separate them; you need churn-DATE data, not a churn flag.")

In [ ]:
# Ex2 - standardise the RM table by CITY instead of segment
rm = cli.groupby("relationship_manager").agg(n=("client_id","count"), churn=("churned","mean"))
city_rates = cli.groupby("city")["churned"].mean()
mix = cli.pivot_table(index="relationship_manager", columns="city", values="client_id", aggfunc="count").fillna(0)
mix = mix.div(mix.sum(axis=1), axis=0)
rm["expected_city"] = (mix*city_rates).sum(axis=1)
print(rm.assign(residual=rm.churn-rm.expected_city).sort_values("residual").round(3).iloc[[0,1,-2,-1]])
print("\nBarely moves: city, like segment, explains little of the RM spread.")
print("Meaning: two measured confounders acquitted. The spread is skill, unmeasured factors, or noise (n~40).")

In [ ]:
# Ex3 - risk_profile within segments
overall = cli.groupby("risk_profile")["churned"].mean()
print((overall*100).round(1), "\n")
piv = cli.pivot_table(index="segment", columns="risk_profile", values="churned")
cnt = cli.pivot_table(index="segment", columns="risk_profile", values="churned", aggfunc="count")
print((piv*100).round(1)); print("\ncounts:\n", cnt)
print("\n'Aggressive is safest' mostly survives within segments too - EXCEPT Ultra-HNI, where the")
print("Aggressive cell is high churn on a tiny count. Who self-selects into 'Aggressive'? Wealthier,")
print("more engaged clients - risk appetite is tangled with segment and engagement. Association, not mechanism.")